### 人口資料與 OHCA 預測值比較

計算 Spearman 等級相關係數，以及 top-10% 高值網格的 Jaccard index。若要納入 SVR，請先在 `SVR_yixing.ipynb` 中輸出 `SVR_prediction.csv`。

In [1]:
import pandas as pd
from functools import reduce
from itertools import combinations
from pathlib import Path
from scipy.stats import spearmanr

# 各資料來源：人口、XGB、MLP；SVR 若有匯出 CSV 也會自動納入
sources = {
    "Population": (["pop_data.csv"], ["pop"]),
    "XGB": (["XGB_prediction.csv"], ["pred_value", "predicted_ohca"]),
    "MLP": (["mlp_prediction.csv"], ["predicted_ohca", "pred_value"]),
    "SVR": (["SVR_prediction.csv", "svr_prediction.csv"], ["predicted_ohca", "pred_value"]),
}

def read_one(name, paths, cols):
    # 讀取第一個存在的檔案，並自動挑選可用的數值欄位
    path = next((Path(p) for p in paths if Path(p).exists()), None)
    if path is None:
        print(f"略過 {name}: 找不到檔案")
        return None
    df = pd.read_csv(path)
    col = next(c for c in cols if c in df.columns)
    return df[["id", col]].rename(columns={col: name}).assign(**{name: lambda x: pd.to_numeric(x[name], errors="coerce")}).dropna()

dfs = [read_one(k, *v) for k, v in sources.items()]
data = reduce(lambda l, r: l.merge(r, on="id", how="inner"), [d for d in dfs if d is not None])
cols = [c for c in data.columns if c != "id"]
print(f"共同網格數 = {data['id'].nunique()}")

共同網格數 = 516


In [2]:
# 兩兩計算 Spearman 等級相關係數
spearman = []
for a, b in combinations(cols, 2):
    x = data[[a, b]].dropna()
    rho, p = spearmanr(x[a], x[b])
    spearman.append({"pair": f"{a} vs {b}", "n": len(x), "rho": rho, "p_value": p})

pd.DataFrame(spearman).sort_values("pair")

,pair,n,rho,p_value
5,MLP vs SVR,516,0.822483,4.738625e-128
1,Population vs MLP,516,0.470698,8.239644e-30
2,Population vs SVR,516,0.676536,2.596593e-70
0,Population vs XGB,516,0.627114,9.487590e-58
3,XGB vs MLP,516,0.536143,9.741694e-40
4,XGB vs SVR,516,0.588642,1.963348e-49


In [3]:
# 取各欄位 top-10% 高值網格，計算高風險區域重疊程度
top = {c: set(data.loc[data[c] >= data[c].quantile(0.9), "id"]) for c in cols}
jaccard = pd.DataFrame(index=cols, columns=cols, dtype=float)

for a in cols:
    for b in cols:
        jaccard.loc[a, b] = len(top[a] & top[b]) / len(top[a] | top[b])

jaccard

,Population,XGB,MLP,SVR
Population,1.000000,0.316456,0.405405,0.424658
XGB,0.316456,1.000000,0.424658,0.424658
MLP,0.405405,0.424658,1.000000,0.824561
SVR,0.424658,0.424658,0.824561,1.000000
